# Stage 4 — Calibrazione k-fold e fusione

Pipeline end-to-end per una singola run **linguaggio / dataset**:

1. carica il parquet *enriched* (`data/enriched/<language>.parquet`) e applica i filtri opzionali;
2. costruisce fold multilabel stratificati sulle CWE (pseudo-label `SAFE` per i campioni non vulnerabili);
3. per ogni fold stima sul *calibration split* le metriche di affidabilità `<tool, cwe>` (`ppv`, `npv`, `fpr`, `fnr`, `supported`);
4. sul *validation split* confronta le strategie di voto **traditional** e **weighted**;
5. salva predizioni, metriche e report (PNG Matplotlib + SVG/HTML) in `data/results/fusion_<language>/`.

**Note.** Il notebook lavora su un solo linguaggio per run. I risultati sono messi in cache:
una chiave di configurazione (`run_config.json`) invalida automaticamente la cache se cambia un
qualsiasi parametro che influenza l'output. La griglia di valutazione delle CWE è controllata da
`EVAL_GRID` (vedi configurazione).

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.dataset import FeastDataset, load_enriched
from analysis.fusion import (
    DEFAULT_WEIGHTED_STRATEGIES,
    all_fusion_predictions,
    cwe_universe,
    evaluate_predictions,
    fireable_cwe_universe,
)
from analysis.metrics import (
    CWERelationshipMatcher,
    compute_tool_cwe_metrics,
    metrics_to_dataframe,
)
from analysis.report import save_fusion_report
from analysis.split import multilabel_stratified_kfold, split_train_validation
from analysis.visualization import (
    plot_metrics_heatmap,
    plot_radar_comparison,
    plot_strategy_comparison,
)

print(f"ROOT: {ROOT}")

## Configurazione

Lascia `LANGUAGE` su un singolo linguaggio (`"python"`, `"java"`, `"c_cpp"`) per evitare ambiguità
nella lookup delle metriche `<tool, cwe>`.

`EVAL_GRID` sceglie l'universo di CWE su cui si valutano le strategie:

- `"fireable"` — solo le CWE che almeno un tool può emettere. Esclude le CWE presenti unicamente
  nelle label (pillar astratti, target di View/Category, debolezze fuori dal catalogo dei tool):
  nessuna strategia potrebbe mai predirle positive, quindi diventerebbero falsi negativi strutturali
  che abbassano artificiosamente recall e F-score.
- `"full"` — universo completo (label + output dei tool).

In [ ]:
LANGUAGE = "python"  # "python" | "java" | "c_cpp"

ENRICHED_DIR = ROOT / "data" / "enriched"
RESULTS_DIR = ROOT / "data" / "results" / f"fusion_{LANGUAGE}"
PLOTS_DIR = RESULTS_DIR / "plots"
CWE_XML_PATH = ROOT / "data" / "cwec_latest.xml"

# Parametri k-fold / fusione
N_SPLITS = 5
RANDOM_STATE = 42
VALIDATION_FOLD = None          # None = tutti i fold; int = solo quel fold
TRADITIONAL_THRESHOLD = 2
EVAL_GRID = "fireable"          # "fireable" | "full"

# Filtri opzionali (None = nessun filtro)
FILTER_BRANCHES = None          # es. "real" oppure ["real", "synth"]
FILTER_DATASETS = None          # es. "PyVul" oppure ["PyVul", "CVEfixes(Python)"]
FILTER_TOOLS = None             # es. "bandit" oppure ["bandit", "semgrep"]
FILTER_CWES = None              # es. ["CWE-20", "CWE-120"]
FILTER_CWES_WHERE = "label"     # "label" | "tools" | "any" | nome tool
MIN_CWE_COUNT = None            # es. 20
MIN_CWE_COUNT_WHERE = "label"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR

## Caricamento e filtri

In [ ]:
dataset = load_enriched(LANGUAGE, enriched_dir=ENRICHED_DIR)

filtered = dataset
if FILTER_BRANCHES is not None:
    filtered = filtered.filter_by_branch(FILTER_BRANCHES)
if FILTER_DATASETS is not None:
    filtered = filtered.filter_by_dataset(FILTER_DATASETS)
if FILTER_TOOLS is not None:
    filtered = filtered.filter_by_tool(FILTER_TOOLS)
if FILTER_CWES is not None:
    filtered = filtered.filter_by_cwe(FILTER_CWES, where=FILTER_CWES_WHERE)
if MIN_CWE_COUNT is not None:
    filtered = filtered.filter_by_min_cwe_count(MIN_CWE_COUNT, where=MIN_CWE_COUNT_WHERE)

if FILTER_TOOLS is None:
    selected_tools = list(filtered.tool_columns)
else:
    requested = FILTER_TOOLS if isinstance(FILTER_TOOLS, list) else [FILTER_TOOLS]
    selected_tools = [str(tool) for tool in requested]

df = filtered.to_pandas().reset_index(drop=True)

print(filtered)
display(pd.DataFrame([filtered.summary()]))
display(filtered.tool_coverage())
display(filtered.cwe_counts_df(where="label", min_count=1).head(25))

## Fold multilabel stratificati

Se la libreria opzionale `iterative-stratification` è installata viene usata direttamente; in
alternativa `analysis.split` applica un fallback deterministico che distribuisce prima le CWE più rare.

La cache dei fold (`folds.csv`) viene riusata solo se i parametri che li determinano
(`language`, numero righe, `N_SPLITS`, `RANDOM_STATE`, filtri) e i `sample_id` coincidono.

In [ ]:
# Chiave di cache: qualunque parametro che influenza i risultati la invalida.
RUN_CONFIG = {
    "language": LANGUAGE,
    "n_rows": int(len(df)),
    "tools": list(selected_tools),
    "n_splits": int(N_SPLITS),
    "random_state": int(RANDOM_STATE),
    "validation_fold": VALIDATION_FOLD,
    "traditional_threshold": int(TRADITIONAL_THRESHOLD),
    "eval_grid": EVAL_GRID,
    "weighted_strategies": [s.name for s in DEFAULT_WEIGHTED_STRATEGIES],
    "filters": {
        "branches": FILTER_BRANCHES,
        "datasets": FILTER_DATASETS,
        "tools": FILTER_TOOLS,
        "cwes": FILTER_CWES,
        "cwes_where": FILTER_CWES_WHERE,
        "min_cwe_count": MIN_CWE_COUNT,
        "min_cwe_count_where": MIN_CWE_COUNT_WHERE,
    },
}

run_config_path = RESULTS_DIR / "run_config.json"
folds_path = RESULTS_DIR / "folds.csv"


def _cached_config():
    if run_config_path.exists():
        try:
            return json.loads(run_config_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            return None
    return None


def _same_sample_ids(a: pd.DataFrame, b: pd.DataFrame) -> bool:
    if len(a) != len(b):
        return False
    if "sample_id" in a.columns and "sample_id" in b.columns:
        return a["sample_id"].astype(str).tolist() == b["sample_id"].astype(str).tolist()
    return True


cached_config = _cached_config()
FOLD_KEYS = ("language", "n_rows", "n_splits", "random_state", "filters")
fold_config_ok = cached_config is not None and all(cached_config.get(k) == RUN_CONFIG[k] for k in FOLD_KEYS)

reuse_folds = False
if fold_config_ok and folds_path.exists():
    cached_folds = pd.read_csv(folds_path)
    reuse_folds = _same_sample_ids(df, cached_folds)

if reuse_folds:
    print(f"Fold caricati dalla cache: {folds_path}")
    folds = cached_folds
else:
    print("Calcolo dei fold stratificati...")
    folds = multilabel_stratified_kfold(df, n_splits=N_SPLITS, random_state=RANDOM_STATE)
    folds.to_csv(folds_path, index=False)

fold_summary = folds["fold"].value_counts().sort_index().rename_axis("fold").reset_index(name="rows")
display(fold_summary)
folds.head()

## Esecuzione k-fold

Per ogni fold:

- **calibration_df** → stima delle metriche di affidabilità `<tool, cwe>` e del flag `supported`;
- **validation_df** → predizioni di fusione e relative performance.

Le coppie `<tool, cwe>` con `supported = False` si astengono nel weighted voting e nelle fusion reliability-based.
I risultati vengono riusati dalla cache solo se l'intera `RUN_CONFIG`
coincide e i `sample_id`/CWE di validation di ogni fold corrispondono.

In [ ]:
target_cwes = (
    fireable_cwe_universe(df, selected_tools)
    if EVAL_GRID == "fireable"
    else cwe_universe(df, selected_tools)
)
print(f"Griglia di valutazione '{EVAL_GRID}': {len(target_cwes)} CWE")

validation_folds = sorted(folds["fold"].unique()) if VALIDATION_FOLD is None else [int(VALIDATION_FOLD)]
matcher = CWERelationshipMatcher.from_xml(CWE_XML_PATH)

calibration_metrics_path = RESULTS_DIR / "calibration_metrics.csv"
predictions_path = RESULTS_DIR / "fusion_predictions.csv"
metrics_by_cwe_path = RESULTS_DIR / "fusion_metrics_by_cwe.csv"
metrics_overall_path = RESULTS_DIR / "fusion_metrics_overall.csv"
result_paths = [calibration_metrics_path, predictions_path, metrics_by_cwe_path, metrics_overall_path]

# I risultati sono compatibili solo se i fold sono stati riusati, la configurazione completa
# coincide e tutti i file esistono. In quel caso verifichiamo anche sample_id/CWE per fold.
reuse_results = reuse_folds and cached_config == RUN_CONFIG and all(p.exists() for p in result_paths)
if reuse_results:
    cached_predictions = pd.read_csv(predictions_path)
    for fold in validation_folds:
        _, validation_df = split_train_validation(df, folds, validation_fold=fold)
        fold_preds = cached_predictions[cached_predictions["fold"] == fold]
        same_ids = set(validation_df["sample_id"].astype(str)) == set(fold_preds["sample_id"].astype(str))
        same_cwes = set(map(str, target_cwes)) == set(fold_preds["cwe"].astype(str))
        if fold_preds.empty or not same_ids or not same_cwes:
            reuse_results = False
            break

if reuse_results:
    print("Risultati caricati dalla cache.")
    calibration_metrics = pd.read_csv(calibration_metrics_path)
    fusion_predictions = pd.read_csv(predictions_path)
    fusion_metrics_by_cwe = pd.read_csv(metrics_by_cwe_path)
    fusion_metrics_overall = pd.read_csv(metrics_overall_path)
else:
    calib_frames, pred_frames, per_cwe_frames, overall_frames = [], [], [], []
    for fold in validation_folds:
        print(f"Fold {fold}: calibrazione -> affidabilità; validation -> confronto fusion")
        calibration_df, validation_df = split_train_validation(df, folds, validation_fold=fold)
        calibration = FeastDataset(
            calibration_df,
            tool_columns=selected_tools,
            history=(*filtered.history, f"calibration fold != {fold}"),
            source_paths=filtered.source_paths,
        )

        metrics_df = metrics_to_dataframe(
            compute_tool_cwe_metrics(
                calibration,
                cwe_xml_path=CWE_XML_PATH,
                tools=selected_tools,
                matcher=matcher,
            )
        )
        metrics_df.insert(0, "fold", fold)
        calib_frames.append(metrics_df)

        predictions = all_fusion_predictions(
            validation_df,
            metrics_df,
            tools=selected_tools,
            cwes=target_cwes,
            matcher=matcher,
            traditional_threshold=TRADITIONAL_THRESHOLD,
            include_supported_traditional=True,
            strategies=DEFAULT_WEIGHTED_STRATEGIES,
        )
        predictions.insert(0, "fold", fold)
        pred_frames.append(predictions)

        per_cwe = evaluate_predictions(predictions, group_cols=("strategy", "cwe"))
        per_cwe.insert(0, "fold", fold)
        per_cwe_frames.append(per_cwe)

        overall = evaluate_predictions(predictions, group_cols=("strategy",))
        overall.insert(0, "fold", fold)
        overall_frames.append(overall)

    calibration_metrics = pd.concat(calib_frames, ignore_index=True)
    fusion_predictions = pd.concat(pred_frames, ignore_index=True)
    fusion_metrics_by_cwe = pd.concat(per_cwe_frames, ignore_index=True)
    fusion_metrics_overall = pd.concat(overall_frames, ignore_index=True)

    calibration_metrics.to_csv(calibration_metrics_path, index=False)
    fusion_predictions.to_csv(predictions_path, index=False)
    fusion_metrics_by_cwe.to_csv(metrics_by_cwe_path, index=False)
    fusion_metrics_overall.to_csv(metrics_overall_path, index=False)
    run_config_path.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
    print(f"Risultati salvati in {RESULTS_DIR}")

## Risultati globali

In [ ]:
metric_columns = ["precision", "recall", "specificity", "f1", "f2", "mcc", "roc_auc", "pr_auc", "balanced_accuracy"]

summary = (
    fusion_metrics_overall
    .groupby("strategy", as_index=False)[metric_columns]
    .mean(numeric_only=True)
    .sort_values("f2", ascending=False, na_position="last")
    .reset_index(drop=True)
)
summary

## Visualizzazioni (Matplotlib → PNG)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# 1. Barre orizzontali per F2
fig, ax = plt.subplots(figsize=(10, 5))
plot_strategy_comparison(fusion_metrics_overall, metric="f2", ax=ax)
fig.savefig(PLOTS_DIR / "strategy_comparison_f2.png", dpi=200, bbox_inches="tight")
plt.show()

# 2. Heatmap delle metriche per strategia
fig, ax = plt.subplots(figsize=(10, 6))
plot_metrics_heatmap(fusion_metrics_overall, ax=ax)
fig.savefig(PLOTS_DIR / "metrics_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

# 3. Radar multidimensionale
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
plot_radar_comparison(fusion_metrics_overall, ax=ax)
fig.savefig(PLOTS_DIR / "strategy_radar.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Grafici PNG salvati in {PLOTS_DIR}")

## Report SVG/HTML

`save_fusion_report` aggrega le metriche globali per strategia e produce grafici SVG leggeri più una
pagina HTML autoconsistente.

In [ ]:
report_paths = save_fusion_report(
    fusion_metrics_overall,
    out_dir=PLOTS_DIR,
    metric_columns=metric_columns,
)
report_paths

In [ ]:
html_path = report_paths["html"]
print(html_path)
display(HTML(html_path.read_text(encoding="utf-8")))

## Confronto per CWE rispetto alla baseline traditional

Delta per la metrica scelta rispetto alla baseline fissa `traditional_<threshold>_of_<N>`.

In [ ]:
DELTA_METRIC = "f2"
BASELINE_STRATEGY = f"traditional_{TRADITIONAL_THRESHOLD}_of_{len(selected_tools)}"

wide = fusion_metrics_by_cwe.pivot_table(
    index=["fold", "cwe"],
    columns="strategy",
    values=DELTA_METRIC,
    aggfunc="mean",
)

if BASELINE_STRATEGY not in wide.columns:
    print(f"Baseline {BASELINE_STRATEGY!r} non trovata. Disponibili: {list(wide.columns)}")
else:
    deltas = wide.subtract(wide[BASELINE_STRATEGY], axis=0).drop(columns=[BASELINE_STRATEGY])
    delta_long = (
        deltas.reset_index()
        .melt(id_vars=["fold", "cwe"], var_name="strategy", value_name=f"delta_{DELTA_METRIC}")
        .dropna()
        .sort_values(f"delta_{DELTA_METRIC}", ascending=False)
    )
    display(delta_long.head(25))
    display(delta_long.tail(25))

## Ispezione del supporto osservato

Coppie `<tool, cwe>` che risultano `unsupported` in **tutti** i fold di calibrazione: durante la
fusione si astengono sempre.

In [ ]:
supported_summary = (
    calibration_metrics
    .groupby(["tool", "cwe"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        supported_folds=("supported", "sum"),
        mean_ppv=("ppv", "mean"),
        mean_npv=("npv", "mean"),
        mean_fpr=("fpr", "mean"),
        mean_fnr=("fnr", "mean"),
    )
)
unsupported = supported_summary[supported_summary["supported_folds"] == 0]
print(f"Unsupported in ogni fold di calibrazione: {len(unsupported):,} coppie tool/CWE")
display(unsupported.head(50))